# 1. Import libaries

In [3]:
import pandas as pd
import numpy as np

# 2. Load raw data

In [2]:
raw_data = pd.read_parquet("data/landing/foodpanda_analysis_dataset_2025.parquet", engine= "pyarrow")
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customer_id      6000 non-null   str    
 1   gender           6000 non-null   str    
 2   age              6000 non-null   str    
 3   city             6000 non-null   str    
 4   signup_date      6000 non-null   str    
 5   order_id         6000 non-null   str    
 6   order_date       6000 non-null   str    
 7   restaurant_name  6000 non-null   str    
 8   dish_name        6000 non-null   str    
 9   category         6000 non-null   str    
 10  quantity         6000 non-null   int64  
 11  price            6000 non-null   float64
 12  payment_method   6000 non-null   str    
 13  order_frequency  6000 non-null   int64  
 14  last_order_date  6000 non-null   str    
 15  loyalty_points   6000 non-null   int64  
 16  churned          6000 non-null   str    
 17  rating           4032 non

In [ ]:
# Preview data values
for col in raw_data.columns:
    print(f"\nColumn: {col}")
    print(f"Number of unique values: {raw_data[col].nunique(dropna=False)}")
    print(raw_data[col].value_counts(dropna=False))


Column: customer_id
Number of unique values: 6000
customer_id
C5663    1
C2831    1
C2851    1
C1694    1
C4339    1
        ..
C6849    1
C3787    1
C2841    1
C1624    1
C2068    1
Name: count, Length: 6000, dtype: int64

Column: gender
Number of unique values: 3
gender
Female    2018
Male      2017
Other     1965
Name: count, dtype: int64

Column: age
Number of unique values: 3
age
Teenager    2062
Adult       1984
Senior      1954
Name: count, dtype: int64

Column: city
Number of unique values: 5
city
Multan       1256
Lahore       1217
Peshawar     1195
Islamabad    1187
Karachi      1145
Name: count, dtype: int64

Column: signup_date
Number of unique values: 730
signup_date
10/18/2023    18
12/30/2024    16
1/26/2025     16
5/25/2025     16
7/18/2025     16
              ..
4/1/2025       2
11/23/2023     2
1/4/2025       2
12/14/2023     2
9/22/2023      2
Name: count, Length: 730, dtype: int64

Column: order_id
Number of unique values: 6000
order_id
O9663     1
O6831     1
O68

# 3. Data cleaning

In [54]:
# Clone df
df = raw_data.copy()

In [55]:
# Convert date cols
date_cols = ["signup_date", "order_date", "last_order_date", "rating_date"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", format="%m/%d/%Y")

# Convert churned to binary
df["is_churned"] = df["churned"].map({"Inactive": 1, "Active": 0})

# Drop old cols
df.drop(columns=["churned"], inplace=True)

# Rename age cols
df.rename(columns={'age': 'age_group'}, inplace=True)

In [56]:
# Preview data values
for col in df.columns:
    print(f"\nColumn: {col}")
    print(f"Number of unique values: {df[col].nunique(dropna=False)}")
    print(df[col].value_counts(dropna=False))


Column: customer_id
Number of unique values: 6000
customer_id
C5663    1
C2831    1
C2851    1
C1694    1
C4339    1
        ..
C6849    1
C3787    1
C2841    1
C1624    1
C2068    1
Name: count, Length: 6000, dtype: int64

Column: gender
Number of unique values: 3
gender
Female    2018
Male      2017
Other     1965
Name: count, dtype: int64

Column: age_group
Number of unique values: 3
age_group
Teenager    2062
Adult       1984
Senior      1954
Name: count, dtype: int64

Column: city
Number of unique values: 5
city
Multan       1256
Lahore       1217
Peshawar     1195
Islamabad    1187
Karachi      1145
Name: count, dtype: int64

Column: signup_date
Number of unique values: 730
signup_date
2023-10-18    18
2024-12-30    16
2025-01-26    16
2025-05-25    16
2025-07-18    16
              ..
2025-04-01     2
2023-11-23     2
2025-01-04     2
2023-12-14     2
2023-09-22     2
Name: count, Length: 730, dtype: int64

Column: order_id
Number of unique values: 6000
order_id
O9663     1
O68

In [57]:
# Check dup record
df.duplicated().sum()

np.int64(0)

> No duplocated records

In [58]:
# Check missing col
print(df['rating'].unique())
df["rating"].describe([.01, .02, .05, .1, .2, .8, .9, .95, .97, .99])

[nan  2.  3.  4.  5.  1.]


count    4032.000000
mean        3.012153
std         1.411615
min         1.000000
1%          1.000000
2%          1.000000
5%          1.000000
10%         1.000000
20%         2.000000
80%         5.000000
90%         5.000000
95%         5.000000
97%         5.000000
99%         5.000000
max         5.000000
Name: rating, dtype: float64

In [64]:
# Fill missing rating with median
df['rating'] = df['rating'].fillna(3)

# Convert rating to int
df['rating'] = df['rating'].astype(int)

In [60]:
# Check invalid transaction
df[(df["quantity"] == 0) | (df["price"] == 0)]

,customer_id,gender,age_group,city,signup_date,order_id,order_date,restaurant_name,dish_name,category,quantity,price,payment_method,order_frequency,last_order_date,loyalty_points,rating,rating_date,delivery_status,is_churned


> No invalid transation

In [61]:
df["value"] = df["quantity"] * df["price"]

In [66]:
df.head()

,customer_id,gender,age_group,city,signup_date,order_id,order_date,restaurant_name,dish_name,category,...,price,payment_method,order_frequency,last_order_date,loyalty_points,rating,rating_date,delivery_status,is_churned,value
0,C5663,Male,Adult,Peshawar,2024-01-14,O9663,2023-08-23,McDonald's,Burger,Italian,...,1478.27,Cash,38,2025-07-19,238,3,2024-10-14,Cancelled,0,7391.35
1,C2831,Male,Adult,Multan,2024-07-07,O6831,2023-08-23,KFC,Burger,Italian,...,956.04,Wallet,24,2024-11-25,81,2,2025-08-21,Delayed,0,2868.12
2,C2851,Other,Senior,Multan,2025-06-20,O6851,2023-08-23,Pizza Hut,Fries,Italian,...,882.51,Cash,42,2025-05-10,82,3,2024-09-19,Delayed,1,1765.02
3,C1694,Female,Senior,Peshawar,2023-09-05,O5694,2023-08-23,Subway,Pizza,Dessert,...,231.30,Card,27,2025-07-24,45,2,2025-06-29,Delayed,1,925.20
4,C4339,Other,Senior,Lahore,2023-12-29,O8339,2023-08-24,KFC,Sandwich,Dessert,...,1156.69,Cash,35,2024-12-21,418,3,2025-03-06,Cancelled,1,1156.69


In [65]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 21 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   customer_id      6000 non-null   str           
 1   gender           6000 non-null   str           
 2   age_group        6000 non-null   str           
 3   city             6000 non-null   str           
 4   signup_date      6000 non-null   datetime64[us]
 5   order_id         6000 non-null   str           
 6   order_date       6000 non-null   datetime64[us]
 7   restaurant_name  6000 non-null   str           
 8   dish_name        6000 non-null   str           
 9   category         6000 non-null   str           
 10  quantity         6000 non-null   int64         
 11  price            6000 non-null   float64       
 12  payment_method   6000 non-null   str           
 13  order_frequency  6000 non-null   int64         
 14  last_order_date  6000 non-null   datetime64[us]
 15

In [67]:
df.to_parquet(
    "data/staging/raw_data_cleaned.parquet",
    index=False,
    engine="pyarrow",
    compression="snappy"
)